# 📏 Project FORESIGHT – Notebook 03: Baseline Models
**NorthBay Living | Demand & Inventory Intelligence**

Establishes baselines using Seasonal Naive and Simple Moving Average.


In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from src.evaluation import compute_all_metrics

DATA = Path('../data/processed')
LAYOUT = dict(paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
HORIZON = 28
print('Ready ✓')

In [ ]:
feat = pd.read_csv(DATA / 'features_engineered.csv', parse_dates=['date'], low_memory=False)

# Filter to SKUs with enough history
counts = feat.groupby('stock_code')['date'].count()
valid_skus = counts[counts >= 90].index
feat = feat[feat['stock_code'].isin(valid_skus)].sort_values(['stock_code','date'])
print(f'Valid SKUs: {len(valid_skus):,}')

## 1. Rolling-Origin Cross Validation Setup

In [ ]:
from src.forecasting import rolling_origin_splits
splits = rolling_origin_splits(feat, n_folds=3, forecast_horizon=HORIZON)
print(f'CV Folds: {len(splits)}')
for i, (train, test) in enumerate(splits):
    print(f'  Fold {i+1}: train={len(train):,} rows ({train["date"].min().date()}→{train["date"].max().date()}) | '
          f'test={len(test):,} rows ({test["date"].min().date()}→{test["date"].max().date()})')

## 2. Seasonal Naive Baseline

In [ ]:
from src.forecasting import SeasonalNaive

naive_metrics_per_fold = []
for fold_idx, (train, test) in enumerate(splits):
    model = SeasonalNaive().fit(train)
    preds = model.predict(test)
    actuals = test['quantity'].clip(lower=0).values
    m = compute_all_metrics(actuals, preds)
    naive_metrics_per_fold.append(m)
    print(f'Fold {fold_idx+1}: WAPE={m["WAPE"]:.2f}%  MAE={m["MAE"]:.2f}  RMSE={m["RMSE"]:.2f}  Bias={m["Bias"]:.2f}')

avg = {k: np.mean([f[k] for f in naive_metrics_per_fold]) for k in naive_metrics_per_fold[0]}
print(f'\n>>> SeasonalNaive AVG: WAPE={avg["WAPE"]:.2f}%  MAE={avg["MAE"]:.2f}  RMSE={avg["RMSE"]:.2f}')

## 3. Moving Average Baseline

In [ ]:
ma_metrics_per_fold = []
for fold_idx, (train, test) in enumerate(splits):
    # Predict using 7-day rolling mean from training set
    sku_ma = train.groupby('stock_code')['quantity'].apply(
        lambda x: x.tail(7).mean()
    ).reset_index()
    sku_ma.columns = ['stock_code','ma_pred']
    test_preds = test.merge(sku_ma, on='stock_code', how='left')
    test_preds['ma_pred'] = test_preds['ma_pred'].fillna(0).clip(lower=0)
    m = compute_all_metrics(test['quantity'].clip(lower=0).values, test_preds['ma_pred'].values)
    ma_metrics_per_fold.append(m)
    print(f'Fold {fold_idx+1}: WAPE={m["WAPE"]:.2f}%  MAE={m["MAE"]:.2f}  RMSE={m["RMSE"]:.2f}')

avg_ma = {k: np.mean([f[k] for f in ma_metrics_per_fold]) for k in ma_metrics_per_fold[0]}
print(f'\n>>> MovingAverage AVG: WAPE={avg_ma["WAPE"]:.2f}%  MAE={avg_ma["MAE"]:.2f}')

## 4. Baseline Comparison

In [ ]:
baseline_comparison = pd.DataFrame([
    {'Model': 'Seasonal Naive (7d)', **avg},
    {'Model': 'Moving Average (7d)', **avg_ma},
])
print(baseline_comparison.to_string(index=False))

fig = go.Figure()
metrics = ['WAPE','MAPE','MAE','RMSE']
colors = ['#6366f1','#06b6d4']
for i, row in baseline_comparison.iterrows():
    fig.add_trace(go.Bar(name=row['Model'],
        x=metrics, y=[row.get(m,0) for m in metrics],
        marker_color=colors[i], text=[f'{row.get(m,0):.2f}' for m in metrics],
        textposition='outside', textfont_color='white'))
fig.update_layout(barmode='group', title='Baseline Model Comparison', height=420, **LAYOUT)
fig.show()

## 5. Sample SKU Baseline Forecast Plot

In [ ]:
# Pick sample SKU and show last fold results
train_last, test_last = splits[-1]
sample = test_last['stock_code'].value_counts().index[0]

train_sku = train_last[train_last['stock_code'] == sample].tail(60)
test_sku  = test_last[test_last['stock_code'] == sample]

naive_m = SeasonalNaive().fit(train_last)
naive_preds_sku = naive_m.predict(test_sku)

fig = go.Figure()
fig.add_trace(go.Scatter(x=train_sku['date'], y=train_sku['quantity'],
    name='Historical', line=dict(color='#06b6d4', width=2)))
fig.add_trace(go.Scatter(x=test_sku['date'], y=test_sku['quantity'],
    name='Actual', line=dict(color='#94a3b8', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=test_sku['date'], y=naive_preds_sku,
    name='Seasonal Naive', line=dict(color='#f59e0b', width=2, dash='dash')))
fig.add_vline(x=test_sku['date'].min(), line_dash='dash', line_color='#6366f1',
    annotation_text='Forecast Start')
fig.update_layout(title=f'Baseline Forecast vs Actual – {sample}', height=400, **LAYOUT)
fig.show()